# FINETUNE: google/vit-base-patch16-224

In [40]:
# Environment Setup and Imports
import os
import random
import numpy as np
from PIL import Image, ImageFile

import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from tqdm import tqdm
from torch import nn, optim

from transformers import ViTModel, ViTFeatureExtractor

# Fix for truncated images loading error
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Device setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

# Dataset path
DATASET_PATH = "/kaggle/input/ai-generated-images-vs-real-images"

Using device: cuda


In [41]:
# Data Preparation with ViT feature extractor and ImageFolder

model_name = "google/vit-base-patch16-224"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

image_size = feature_extractor.size["height"]  # typically 224

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std),
])


# def is_valid_image(path):
#     """Checks if a file is a valid, openable image."""
#     try:
#         with Image.open(path) as img:
#             img.verify()  # Verify the image data
#         return True
#     except Exception as e:
#         print(f"Skipping corrupt image: {path}, error: {e}")
#         return False


# train_dataset = datasets.ImageFolder(
#     root=f"{DATASET_PATH}/train",
#     transform=train_transform,
#     is_valid_file=is_valid_image,
# )

# val_dataset = datasets.ImageFolder(
#     root=f"{DATASET_PATH}/test",
#     transform=val_transform,
#     is_valid_file=is_valid_image,
# )

train_dataset = datasets.ImageFolder(
    root=f"{DATASET_PATH}/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=f"{DATASET_PATH}/test",
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.class_to_idx}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


Train samples: 48000
Val samples: 12000
Classes: {'fake': 0, 'real': 1}


In [42]:
# Define Model Architecture with ViT backbone + classification head

class ViTClassifier(nn.Module):
    def __init__(self, model_name="google/vit-base-patch16-224"):
        super(ViTClassifier, self).__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        
        # Freeze ViT backbone parameters (optional)
        for param in self.vit.parameters():
            param.requires_grad = False
        
        hidden_size = self.vit.config.hidden_size  # typically 768
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),  # Binary classification
        )
    
    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
        pooled_output = outputs.pooler_output  # shape (batch_size, hidden_size)
        logits = self.classifier(pooled_output)
        return logits.squeeze(-1)


model = ViTClassifier().to(DEVICE)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of the model checkpoint at google/vit-base-patch16-224 were not used when initializing ViTModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [43]:
# Step 4: Loss and optimizer

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)

In [44]:
# Train for one epoch

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(dataloader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(dataloader)

In [45]:
# Validate model accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device).float()

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    avg_loss = running_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy


In [17]:
# Evaluate BEFORE fine-tuning
initial_val_loss, initial_val_acc = evaluate(model, val_loader, criterion, DEVICE)
print(f"Initial Validation Loss: {initial_val_loss:.4f}")
print(f"Initial Validation Accuracy: {initial_val_acc:.4f}")

Evaluating:   6%|▌         | 23/375 [00:12<02:44,  2.14it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Evaluating:  15%|█▍        | 56/375 [00:28<02:22,  2.24it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Evaluating:  17%|█▋        | 63/375 [00:31<02:16,  2.28it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Evaluating:  64%|██████▍   | 240/375 [03:38<10:28,  4.65s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Evaluating:  64%|██████▍ 

Initial Validation Loss: 0.6976
Initial Validation Accuracy: 0.5007


In [ ]:
# Evaluation Results BEFORE Finetuning
# Initial Validation Loss: 0.6976
# Initial Validation Accuracy: 0.5007

In [46]:
# Full training + validation loop

best_val_accuracy = 0.0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    print(f"Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Validation Loss: {val_loss:.4f}")
    print(f"  Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), "best_vit_classifier.pth")
        print(f"🎉 New best model saved! Accuracy: {val_acc:.4f} at epoch {epoch+1}")

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")


--- Epoch 1/3 ---


Training:   4%|▍         | 61/1500 [01:21<21:37,  1.11it/s]  /usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99991727 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:  13%|█▎        | 188/1500 [04:01<33:37,  1.54s/it]  /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  21%|██▏       | 322/1500 [06:52<17:16,  1.14it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:  22%|██▏       | 334/1500 [07:07<17:14,  1.13it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.w

Epoch 1 Summary:
  Train Loss: 0.4138
  Validation Loss: 0.3285
  Validation Accuracy: 0.8570
🎉 New best model saved! Accuracy: 0.8570 at epoch 1

--- Epoch 2/3 ---


Training:   1%|          | 10/1500 [00:15<24:12,  1.03it/s] /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   4%|▍         | 61/1500 [01:21<25:56,  1.08s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   6%|▌         | 89/1500 [01:57<29:56,  1.27s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:   7%|▋         | 103/1500 [02:11<12:42,  1.83it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  12%|█▏        | 17

Epoch 2 Summary:
  Train Loss: 0.3232
  Validation Loss: 0.2951
  Validation Accuracy: 0.8716
🎉 New best model saved! Accuracy: 0.8716 at epoch 2

--- Epoch 3/3 ---


Training:   3%|▎         | 43/1500 [00:55<24:28,  1.01s/it] /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  11%|█▏        | 171/1500 [03:39<15:32,  1.43it/s]  /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  12%|█▏        | 180/1500 [03:50<16:17,  1.35it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:  14%|█▎        | 204/1500 [04:26<23:29,  1.09s/it]  /usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99991727 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings

Epoch 3 Summary:
  Train Loss: 0.2938
  Validation Loss: 0.2780
  Validation Accuracy: 0.8812
🎉 New best model saved! Accuracy: 0.8812 at epoch 3

Training complete.
Best validation accuracy: 0.8812


In [ ]:
# Evaluation Results AFTER Finetuning

# Epoch 1 Summary:
#   Train Loss: 0.4138
#   Validation Loss: 0.3285
#   Validation Accuracy: 0.8570

# Epoch 2 Summary:
#   Train Loss: 0.3232
#   Validation Loss: 0.2951
#   Validation Accuracy: 0.8716

# Epoch 3 Summary:
#   Train Loss: 0.2938
#   Validation Loss: 0.2780
#   Validation Accuracy: 0.8812

# FINETUNE: openai/clip-vit-base-patch16

In [3]:
# Environment Setup and Imports
import os
import random
import numpy as np
from PIL import Image, ImageFile

import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from tqdm import tqdm
from torch import nn, optim

from transformers import CLIPModel, CLIPImageProcessor

# Fix for truncated images loading error
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Device setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

# Hyperparameters
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

# Dataset path
DATASET_PATH = "/kaggle/input/ai-generated-images-vs-real-images"

2025-11-07 18:57:11.517580: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762541831.712791      70 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762541831.766362      70 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda


In [4]:
# Data Preparation with CLIP Image Processor and ImageFolder

clip_model_name = "openai/clip-vit-base-patch16"
processor = CLIPImageProcessor.from_pretrained(clip_model_name)

image_size = (
    processor.size.get("shortest_edge", 224)
    if isinstance(processor.size, dict)
    else processor.size
)

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std),
])

# def is_valid_image(path):
#     """Checks if a file is a valid, openable image."""
#     try:
#         with Image.open(path) as img:
#             img.verify()  # Verify the image data
#         return True
#     except Exception as e:
#         print(f"Skipping corrupt image: {path}, error: {e}")
#         return False

# train_dataset = datasets.ImageFolder(
#     root=f"{DATASET_PATH}/train", 
#     transform=train_transform,
#     is_valid_file=is_valid_image
# )

# val_dataset = datasets.ImageFolder(
#     root=f"{DATASET_PATH}/test", 
#     transform=val_transform,
#     is_valid_file=is_valid_image
# )

train_dataset = datasets.ImageFolder(
    root=f"{DATASET_PATH}/train", 
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=f"{DATASET_PATH}/test", 
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=4, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Classes: {train_dataset.class_to_idx}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

Train samples: 48000
Val samples: 12000
Classes: {'fake': 0, 'real': 1}


In [5]:
# Define Model Architecture with classification head (no sigmoid)

class CLIPImageClassifier(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch16"):
        super(CLIPImageClassifier, self).__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)
        
        # Freeze CLIP parameters so we only train the classifier head
        for param in self.clip.parameters():
            param.requires_grad = False
        
        self.classifier = nn.Sequential(
            nn.Linear(self.clip.config.vision_config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),  # Single output logit for binary classification
        )
    
    def forward(self, pixel_values):
        vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
        image_features = vision_outputs.pooler_output
        logits = self.classifier(image_features)
        return logits.squeeze(-1)  # Output shape: [batch_size]

model = CLIPImageClassifier().to(DEVICE)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

In [6]:
# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=LEARNING_RATE)

In [7]:
# Train for one epoch

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(dataloader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device).float()  # float required for BCEWithLogitsLoss
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(dataloader)

In [8]:
# Validate model accuracy

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device)
            labels = labels.to(device).float()
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    avg_loss = running_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

In [25]:
# Evaluate BEFORE fine-tuning
initial_val_loss, initial_val_acc = evaluate(model, val_loader, criterion, DEVICE)
print(f"Initial Validation Loss: {initial_val_loss:.4f}")
print(f"Initial Validation Accuracy: {initial_val_acc:.4f}")

Evaluating:   7%|▋         | 25/375 [00:12<02:08,  2.72it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Evaluating:  15%|█▍        | 56/375 [00:26<02:35,  2.05it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Evaluating:  18%|█▊        | 67/375 [00:31<01:54,  2.68it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Evaluating:  64%|██████▍   | 241/375 [03:17<08:36,  3.85s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.11

Initial Validation Loss: 0.7039
Initial Validation Accuracy: 0.4785


In [ ]:
# Evaluation Results BEFORE Finetuning
# Initial Validation Loss: 0.7039
# Initial Validation Accuracy: 0.4785

In [9]:
# Full training + validation loop

best_val_accuracy = 0.0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    
    print(f"Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Validation Loss: {val_loss:.4f}")
    print(f"  Validation Accuracy: {val_acc:.4f}")
    
    # Save model checkpoint if improvement in validation accuracy
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), "best_clip_vit_base_patch16_classifier.pth")
        print(f"🎉 New best model saved! Accuracy: {val_acc:.4f} at epoch {epoch+1}")

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")


--- Epoch 1/3 ---


Training:   3%|▎         | 52/1500 [00:59<18:26,  1.31it/s] /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   7%|▋         | 103/1500 [01:57<22:26,  1.04it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   8%|▊         | 123/1500 [02:17<19:12,  1.19it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  30%|███       | 456/1500 [08:50<12:29,  1.39it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (161087488 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:  36%|███▌      |

Epoch 1 Summary:
  Train Loss: 0.1932
  Validation Loss: 0.1282
  Validation Accuracy: 0.9503
🎉 New best model saved! Accuracy: 0.9503 at epoch 1

--- Epoch 2/3 ---


Training:   1%|          | 15/1500 [00:21<35:07,  1.42s/it] /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   6%|▋         | 97/1500 [01:46<20:22,  1.15it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:   7%|▋         | 106/1500 [01:55<16:36,  1.40it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (99991727 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:  10%|█         | 151/1500 [02:45<13:07,  1.71it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb

Epoch 2 Summary:
  Train Loss: 0.1264
  Validation Loss: 0.1112
  Validation Accuracy: 0.9580
🎉 New best model saved! Accuracy: 0.9580 at epoch 2

--- Epoch 3/3 ---


Training:   0%|          | 6/1500 [00:09<32:24,  1.30s/it]  /usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   1%|▏         | 22/1500 [00:27<19:15,  1.28it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   4%|▍         | 62/1500 [01:13<20:15,  1.18it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:   4%|▍         | 66/1500 [01:17<18:16,  1.31it/s]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  25%|██▍       | 368/1500 [06:43<18:09,  1.04it/s

Epoch 3 Summary:
  Train Loss: 0.1097
  Validation Loss: 0.1016
  Validation Accuracy: 0.9611
🎉 New best model saved! Accuracy: 0.9611 at epoch 3

Training complete.
Best validation accuracy: 0.9611


In [ ]:
# Evaluation Results AFTER Finetuning

# Epoch 1 Summary:
#   Train Loss: 0.1932
#   Validation Loss: 0.1282
#   Validation Accuracy: 0.9503

# Epoch 2 Summary:
#   Train Loss: 0.1264
#   Validation Loss: 0.1112
#   Validation Accuracy: 0.9580

# Epoch 3 Summary:
#   Train Loss: 0.1097
#   Validation Loss: 0.1016
#   Validation Accuracy: 0.9611

In [ ]:
# # Load Finetuned Model
# FINETUNED_MODEL_NAME = "best_convnextv2.pth"

# # Step 1: Recreate the model architecture
# model = ConvNeXtV2ImageClassifier().to(DEVICE)

# # Step 2: Load the saved weights
# model.load_state_dict(torch.load(f"/kaggle/working/{FINETUNED_MODEL_NAME}", map_location=DEVICE))

# # Step 3: Set model to evaluation mode
# model.eval()

# # Step 4: Evaluate on validation set
# val_loss, val_accuracy = evaluate(model, val_loader, criterion, DEVICE)

# print(f"Loaded Model Validation Loss: {val_loss:.4f}")
# print(f"Loaded Model Validation Accuracy: {val_accuracy:.4f}")